# Практическое занятие 3.3.2 (ПЗ-10)
## Реализация упрощённого offline/online-пайплайна обработки данных и автоматизация вычислительного процесса

**Дисциплина:** Системы обработки больших данных  
**Студент:** ____________________  
**Группа:** _____________________  
**Вариант:** ____________________  
**Дата:** _______________________  

Выполняйте работу по своему индивидуальному варианту. Ноутбук и датасеты должны лежать **в одной папке**.


## Цель работы
Реализовать учебный пайплайн обработки данных, включающий загрузку, очистку, преобразование, интеграцию, batch-агрегацию, итоговую витрину, упрощённую event-like обработку и описание логики оркестрации.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, Markdown

plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)


In [ ]:
# Справочник вариантов и рекомендуемые имена файлов датасетов
variant_catalog = {
    "1": "Банковские транзакции и антифрод-мониторинг",
    "2": "Персонализация предложений маркетплейса",
    "3": "Логи цифровой образовательной платформы",
    "4": "Телеметрия промышленного оборудования",
    "5": "Обращения граждан в цифровую приемную",
    "6": "Маршруты городского общественного транспорта",
    "7": "Заказы и сроки доставки в логистике",
    "8": "Клиентская активность оператора связи",
    "9": "Медицинские записи и результаты обследований",
    "10": "События мобильного банковского приложения",
    "11": "Продажи и посещаемость розничной сети",
    "12": "Заявки службы технической поддержки",
    "13": "Публикации и активность в социальной платформе",
    "14": "Данные энергопотребления объекта",
    "15": "Дорожная обстановка и транспортные события",
    "16": "Сельскохозяйственные данные и мониторинг полей",
    "17": "Транзакции и действия пользователей интернет-магазина",
    "18": "Документы и контракты организации",
    "19": "Журналы киберинцидентов и событий безопасности",
    "20": "Туристические потоки и загрузка инфраструктуры региона"
}

variant_number = 1  # <-- студент меняет номер своего варианта
variant_key = str(variant_number)

base_path = Path(".")  # ноутбук и данные должны лежать в одной папке

main_file = base_path / f"variant_{variant_number:02d}_main.csv"
reference_file = base_path / f"variant_{variant_number:02d}_reference.csv"
stream_file = base_path / f"variant_{variant_number:02d}_stream.csv"

display(Markdown(f"### Вариант {variant_number}"))
display(Markdown(variant_catalog[variant_key]))
display(Markdown(f"**Основной файл:** `{main_file.name}`  
**Справочный файл:** `{reference_file.name}`  
**Файл для event-like обработки:** `{stream_file.name}`"))


## 1. Постановка индивидуального задания
**Формулировка варианта:**  
_Кратко перепишите текст своего варианта._

**Предметная область:**  
_Опишите предметную область в 1–2 абзацах._

**Цель пайплайна:**  
_Укажите, какой аналитический результат должен формироваться на выходе._


## 2. Задачи работы
Сформулируйте задачи для своего варианта. Рекомендуемый состав:

1. Загрузить и описать исходные данные.  
2. Выполнить первичный анализ качества данных.  
3. Очистить и подготовить данные.  
4. Объединить источники данных при необходимости.  
5. Построить batch-пайплайн.  
6. Сформировать итоговую витрину.  
7. Реализовать event-like обработку.  
8. Описать логику оркестрации и автоматизации.


In [ ]:
# Загрузка данных
# Если справочный или stream-файл отсутствует, соответствующая переменная останется None.

df_main = pd.read_csv(main_file)
df_ref = pd.read_csv(reference_file) if reference_file.exists() else None
df_stream = pd.read_csv(stream_file) if stream_file.exists() else None

print("Основной набор данных:", df_main.shape)
display(df_main.head())

if df_ref is not None:
    print("Справочный набор данных:", df_ref.shape)
    display(df_ref.head())

if df_stream is not None:
    print("Потоковый (event-like) набор данных:", df_stream.shape)
    display(df_stream.head())


## 3. Первичный анализ исходных данных
После загрузки данных кратко опишите:

- какие сущности представлены в данных;
- какие поля являются ключевыми;
- какие поля содержат время, идентификаторы, категории, числовые значения;
- какие проблемы качества заметны сразу.


In [ ]:
# Первичный анализ основного набора
df_main.info()
display(df_main.isna().sum().to_frame("missing_values"))
display(pd.DataFrame({"duplicate_rows": [df_main.duplicated().sum()]}))
display(df_main.describe(include="all").T.head(20))


## 4. Настройка ролей столбцов
В этой ячейке укажите, какие столбцы вашего датасета играют роль времени, идентификатора, категории, числового значения и ключа для объединения.

Если некоторого столбца нет, оставьте значение `None`.


In [ ]:
# Настройте имена столбцов под свой вариант
time_col = None          # например: "timestamp"
entity_id_col = None     # например: "user_id", "device_id", "order_id"
category_col = None      # например: "category", "event_type", "region"
value_col = None         # например: "amount", "duration", "consumption"
status_col = None        # например: "status"
join_key = None          # например: "user_id", "route_id", "product_id"

# Пример:
# time_col = "timestamp"
# entity_id_col = "user_id"
# category_col = "event_type"
# value_col = "amount"
# join_key = "user_id"


In [ ]:
# Проверка существования выбранных столбцов
selected_columns = {
    "time_col": time_col,
    "entity_id_col": entity_id_col,
    "category_col": category_col,
    "value_col": value_col,
    "status_col": status_col,
    "join_key": join_key,
}

for role, col_name in selected_columns.items():
    if col_name is not None:
        exists = col_name in df_main.columns or (df_ref is not None and col_name in df_ref.columns)
        print(f"{role}: {col_name} ->", "OK" if exists else "ПРОВЕРИТЬ")


## 5. Очистка и подготовка данных
На этом этапе необходимо:

- обработать пропуски;
- удалить дубликаты;
- привести типы данных;
- нормализовать категориальные значения;
- преобразовать время в формат `datetime`, если в данных есть временные метки.


In [ ]:
def clean_data(df: pd.DataFrame, time_col=None):
    result = df.copy()

    # Удаление полных дубликатов
    result = result.drop_duplicates()

    # Приведение строковых значений к единому виду
    for col in result.select_dtypes(include="object").columns:
        result[col] = result[col].astype(str).str.strip()

    # Преобразование времени
    if time_col is not None and time_col in result.columns:
        result[time_col] = pd.to_datetime(result[time_col], errors="coerce")

    return result

clean_main = clean_data(df_main, time_col=time_col)
clean_ref = clean_data(df_ref, time_col=None) if df_ref is not None else None
clean_stream = clean_data(df_stream, time_col=time_col) if df_stream is not None else None

print("После очистки основной набор:", clean_main.shape)
display(clean_main.head())


In [ ]:
# Контроль качества после очистки
display(clean_main.isna().sum().to_frame("missing_after_cleaning"))
display(pd.DataFrame({"duplicate_rows_after_cleaning": [clean_main.duplicated().sum()]}))


**Краткий вывод по очистке:**  
_Опишите, какие именно проблемы были устранены и как это повлияло на данные._


## 6. Интеграция источников данных
Если в варианте есть справочный набор данных, выполните объединение по ключевому полю. Если справочного файла нет, кратко объясните, почему интеграция не требуется.


In [ ]:
if clean_ref is not None and join_key is not None and join_key in clean_main.columns and join_key in clean_ref.columns:
    df_work = clean_main.merge(clean_ref, on=join_key, how="left")
else:
    df_work = clean_main.copy()

print("Рабочий набор данных:", df_work.shape)
display(df_work.head())


**Краткий вывод по интеграции:**  
_Укажите, по какому ключу выполнялось объединение, были ли потери строк, как изменилась структура рабочего набора._


## 7. Преобразование и построение признаков
Постройте производные признаки, которые понадобятся для агрегации и анализа. Например:

- дата без времени;
- час события;
- день недели;
- признак выходного дня;
- длительность или числовой показатель;
- укрупнённая категория.


In [ ]:
def transform_data(df: pd.DataFrame, time_col=None):
    result = df.copy()

    if time_col is not None and time_col in result.columns:
        result["event_datetime"] = pd.to_datetime(result[time_col], errors="coerce")
        result["event_date"] = result["event_datetime"].dt.date
        result["event_hour"] = result["event_datetime"].dt.hour
        result["event_weekday"] = result["event_datetime"].dt.weekday
        result["is_weekend"] = result["event_weekday"] >= 5

    return result

df_transformed = transform_data(df_work, time_col=time_col)
display(df_transformed.head())


**Краткий вывод по признакам:**  
_Перечислите созданные признаки и поясните, зачем они нужны для последующего анализа._


## 8. Реализация batch-пайплайна
Ниже batch-версия пайплайна оформляется как последовательность этапов. Важно показать, что обработка данных выполняется как структурированный процесс.


In [ ]:
def load_data_pipeline(main_path, ref_path=None, stream_path=None):
    main = pd.read_csv(main_path)
    ref = pd.read_csv(ref_path) if ref_path is not None and Path(ref_path).exists() else None
    stream = pd.read_csv(stream_path) if stream_path is not None and Path(stream_path).exists() else None
    return main, ref, stream

def prepare_main_pipeline(main_df, ref_df=None, time_col=None, join_key=None):
    main_clean = clean_data(main_df, time_col=time_col)
    ref_clean = clean_data(ref_df) if ref_df is not None else None

    if ref_clean is not None and join_key is not None and join_key in main_clean.columns and join_key in ref_clean.columns:
        merged = main_clean.merge(ref_clean, on=join_key, how="left")
    else:
        merged = main_clean.copy()

    transformed = transform_data(merged, time_col=time_col)
    return transformed

def build_batch_report(df, category_col=None, value_col=None):
    result = df.copy()

    group_cols = []
    if "event_date" in result.columns:
        group_cols.append("event_date")
    if category_col is not None and category_col in result.columns:
        group_cols.append(category_col)

    if not group_cols:
        report = pd.DataFrame({"event_count": [len(result)]})
    else:
        if value_col is not None and value_col in result.columns and pd.api.types.is_numeric_dtype(result[value_col]):
            report = (
                result.groupby(group_cols)
                .agg(event_count=(result.columns[0], "count"),
                     value_sum=(value_col, "sum"),
                     value_mean=(value_col, "mean"))
                .reset_index()
            )
        else:
            report = (
                result.groupby(group_cols)
                .agg(event_count=(result.columns[0], "count"))
                .reset_index()
            )
    return report

main_loaded, ref_loaded, stream_loaded = load_data_pipeline(
    main_file,
    reference_file if reference_file.exists() else None,
    stream_file if stream_file.exists() else None
)

df_batch_ready = prepare_main_pipeline(
    main_loaded,
    ref_loaded,
    time_col=time_col,
    join_key=join_key
)

batch_report = build_batch_report(
    df_batch_ready,
    category_col=category_col,
    value_col=value_col
)

print("Итоговая batch-витрина:", batch_report.shape)
display(batch_report.head(20))


**Краткий вывод по batch-пайплайну:**  
_Опишите, какие этапы он включает, какие агрегаты формируются и почему этот результат можно считать итоговой аналитической витриной._


In [ ]:
# При необходимости сохраните витрину в файл
batch_report.to_csv(f"variant_{variant_number:02d}_batch_report.csv", index=False, encoding="utf-8-sig")
print("Файл batch-отчёта сохранён.")


## 9. Визуализация batch-результата
Постройте 1–3 осмысленных графика на основе итоговой витрины. Графики должны помогать интерпретировать результат.


In [ ]:
if "event_date" in batch_report.columns and "event_count" in batch_report.columns:
    plot_df = batch_report.groupby("event_date", as_index=False)["event_count"].sum()
    plt.figure()
    plt.plot(plot_df["event_date"].astype(str), plot_df["event_count"])
    plt.xticks(rotation=45)
    plt.title("Динамика числа событий по датам")
    plt.xlabel("Дата")
    plt.ylabel("Количество событий")
    plt.tight_layout()
    plt.show()
elif category_col is not None and category_col in batch_report.columns and "event_count" in batch_report.columns:
    plot_df = batch_report.groupby(category_col, as_index=False)["event_count"].sum()
    plt.figure()
    plt.bar(plot_df[category_col].astype(str), plot_df["event_count"])
    plt.xticks(rotation=45)
    plt.title("Распределение событий по категориям")
    plt.xlabel("Категория")
    plt.ylabel("Количество событий")
    plt.tight_layout()
    plt.show()
else:
    print("Подберите визуализацию вручную под структуру своего варианта.")


**Интерпретация визуализации:**  
_Кратко поясните, что показывает построенный график и какие выводы можно сделать._


## 10. Event-like обработка (упрощённая online-имитация)
Задача этого этапа — показать, как изменилась бы логика работы, если бы события поступали последовательно. При отсутствии отдельного stream-файла используйте основной набор данных.


In [ ]:
df_event = clean_stream.copy() if clean_stream is not None else clean_main.copy()

if time_col is not None and time_col in df_event.columns:
    df_event[time_col] = pd.to_datetime(df_event[time_col], errors="coerce")
    df_event = df_event.sort_values(time_col).reset_index(drop=True)
else:
    df_event = df_event.reset_index(drop=True)

display(df_event.head())
print("Размер набора для event-like обработки:", df_event.shape)


In [ ]:
# Простая event-like модель: накопительный счётчик событий
df_event_like = df_event.copy()
df_event_like["event_sequence_number"] = range(1, len(df_event_like) + 1)
df_event_like["rolling_count_last_10_events"] = (
    df_event_like["event_sequence_number"]
    .rolling(window=10, min_periods=1)
    .count()
)

display(df_event_like.head(20))


In [ ]:
plt.figure()
plt.plot(df_event_like["event_sequence_number"], df_event_like["rolling_count_last_10_events"])
plt.title("Упрощённая event-like обработка: скользящее окно по 10 последним событиям")
plt.xlabel("Порядковый номер события")
plt.ylabel("Количество событий в окне")
plt.tight_layout()
plt.show()


**Краткий вывод по event-like обработке:**  
_Поясните, чем такой подход отличается от batch-обработки и в чём его практический смысл._


## 11. Сравнение batch и event-like подходов
Сравните два режима обработки по следующим критериям:

- полнота данных;
- скорость получения результата;
- удобство агрегации;
- пригодность для отчётности;
- пригодность для оперативного реагирования.


In [ ]:
comparison_df = pd.DataFrame({
    "Критерий": [
        "Полнота данных",
        "Скорость получения результата",
        "Удобство агрегации",
        "Пригодность для отчётности",
        "Пригодность для оперативного реагирования"
    ],
    "Batch": [""] * 5,
    "Event-like": [""] * 5
})
comparison_df


## 12. Логика оркестрации
Даже если вы не используете реальный оркестратор, необходимо явно показать порядок выполнения задач, зависимости и действия при ошибке.


In [ ]:
orchestration_df = pd.DataFrame({
    "Задача": [
        "Загрузка данных",
        "Очистка данных",
        "Интеграция источников",
        "Преобразование признаков",
        "Построение batch-витрины",
        "Event-like обработка",
        "Публикация результата"
    ],
    "От каких задач зависит": [
        "",
        "Загрузка данных",
        "Очистка данных",
        "Интеграция источников",
        "Преобразование признаков",
        "Очистка данных",
        "Построение batch-витрины"
    ],
    "Тип запуска": [""] * 7,
    "Что делать при ошибке": [""] * 7
})
orchestration_df


**Краткий вывод по оркестрации:**  
_Опишите, какие этапы должны запускаться по расписанию, какие могут запускаться по событию и почему ручной запуск неудобен._


## 13. Схема учебного пайплайна
Вставьте сюда схему реализованного учебного пайплайна.

Пример вставки:  
`![Схема пайплайна](pipeline_scheme.png)`

На схеме желательно показать:

- входные данные;
- очистку;
- интеграцию;
- преобразование;
- batch-агрегацию;
- event-like обработку;
- итоговую витрину;
- условный блок оркестрации.


## 14. Итоговые выводы
В выводах отразите:

1. Какой пайплайн был реализован.  
2. Какие этапы оказались обязательными.  
3. Что показала batch-обработка.  
4. Что показала event-like обработка.  
5. Какой итоговый аналитический результат был построен.  
6. Почему этот notebook можно считать учебной моделью промышленного пайплайна.  
7. Что оказалось самым сложным в работе.


## 15. Контрольные вопросы
1. Что такое batch-пайплайн?  
_Ответ._

2. Что такое near-real-time или event-like обработка?  
_Ответ._

3. Почему этап очистки должен быть отдельным этапом пайплайна?  
_Ответ._

4. Почему итоговая витрина является конечным результатом пайплайна?  
_Ответ._

5. Что даёт оркестрация задач?  
_Ответ._

6. Чем Jupyter Notebook в этой работе похож на модель промышленного пайплайна?  
_Ответ._


## 16. Чек-лист готовности
- [ ] Указан номер варианта  
- [ ] Загружены исходные данные  
- [ ] Проведён первичный анализ  
- [ ] Выполнена очистка данных  
- [ ] Выполнена интеграция источников  
- [ ] Построены производные признаки  
- [ ] Реализован batch-пайплайн  
- [ ] Построена итоговая витрина  
- [ ] Выполнена event-like обработка  
- [ ] Сравнены batch и event-like подходы  
- [ ] Описана логика оркестрации  
- [ ] Добавлена схема пайплайна  
- [ ] Сформулированы выводы  
- [ ] Даны ответы на контрольные вопросы
